[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_2_Lagrange_Method_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


<div dir="rtl" style="text-align:right">

# 🧭 اللاب 2: المعادلات شبه الخطية وطريقة لاجرانج لخطوط الانسياب
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في ملاحة المركبات الفضائية  
**المرجع في المذكرة:** الباب الأول، القسم 1.1.7 (ص 4–5) والقسم 1.3.1–1.3.2 (صفحات المذكرة 11–16 / صفحات PDF: 12–17)

> 🗺️ **أين يقع هذا اللاب في خط عمل المشروع؟**  
> `خريطة الارتفاعات` $\longrightarrow$ `كشف العوائق` $\longrightarrow$ `نطاق الأمان` $\longrightarrow$ `معادلة لابلاس φ(x,y)` $\longrightarrow$ **`[اللاب 2: تتبع خط الانسياب descend()]`**  
> **الملف الحقيقي في المشروع:** [`rover_path_finder/planners.py`](../rover_path_finder/planners.py) — الدالة `descend(P, hazard, start, goal)`

---
### 🧭 قاموس الربط: الرياضيات $\longleftrightarrow$ المعنى الفيزيائي $\longleftrightarrow$ كود المشروع
| 1. الرمز في مذكرة المحاضرة | 2. المعنى الفيزيائي على أرض المريخ | 3. الكود الفعلي في `planners.py` |
| :--- | :--- | :--- |
| $P(x,y,z)\,p + Q(x,y,z)\,q = R(x,y,z)$ | **معادلة لاجرانج شبه الخطية:** المشتقتان $p, q$ من الدرجة الأولى | $\phi_x \frac{\partial\psi}{\partial x} + \phi_y \frac{\partial\psi}{\partial y} = 0$ (معادلة دالة الانسياب $\psi$) |
| $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$ | **معادلات لاجرانج المساعدة:** معادلات المسار الذي يسلكه الجسيم خطوة بخطوة | $\frac{dx}{-\phi_x} = \frac{dy}{-\phi_y}$ (السير مع أشد انحدار) |
| $u(x,y,z) = a, \quad v(x,y,z) = b$ | ثابتان يبقيان محفوظين طوال حركة المركبة على نفس المسار | منحنى مسار المركبة `path = [(x, y), ...]` |
| $\Phi(u, v) = 0$ | **الحل العام:** يجمع كل المسارات الممكنة بدالة اختيارية $\Phi$ | عائلة خطوط الانسياب الكاملة من البداية للهدف |

---
## 1. الفكرة ببساطة: كيف تتحول معادلة تفاضلية جزئية إلى مسار قيادة؟
بعد أن يحسب حاسوب المركبة حقل الجهد التوافقي $\phi(x, y)$ (حيث البداية عند $\phi=1$ والهدف عند $\phi=0$)، كيف تقود المركبة عجلاتها؟
المركبة تتجه في كل لحظة باتجاه **أشد انحدار للأسفل (Steepest Descent)**:
$$\mathbf{v} = (u, v) = -\nabla\phi = \left(-\frac{\partial\phi}{\partial x},\; -\frac{\partial\phi}{\partial y}\right)$$
إذا أردنا إيجاد **دالة الانسياب $\psi(x, y)$** التي تبقى قيمتها ثابتة على طول مسار المركبة ($d\psi = 0$ على المسار)، فإن $\psi$ تحقق معادلة لاجرانج الخطية من الرتبة الأولى:
$$u \frac{\partial\psi}{\partial x} + v \frac{\partial\psi}{\partial y} = 0 \qquad \text{أي أن:} \qquad P = u,\; Q = v,\; R = 0$$
وعندما نكتب **معادلات لاجرانج المساعدة** لهذه المعادلة نحصل على:
$$\frac{dx}{u} = \frac{dy}{v} = \frac{d\psi}{0} \implies \psi(x,y) = \text{ثابت على طول مسار المركبة!}$$
إذن: **المنحنيات المميزة لطريقة لاجرانج (Characteristic Curves) هي بالضبط المسارات الهندسية التي تسير عليها عجلات المركبة في دالة `descend()`!**

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab2

%matplotlib inline
print("تم تهيئة بيئة اللاب 2 بنجاح!")


<div dir="rtl" style="text-align:right">

---
## 2. قاعدة العمل والاستراتيجيات الثلاث لحل معادلات لاجرانج في المذكرة (PDF ص 12–17)

### 🎯 كيف أميّز معادلة لاجرانج في الامتحان؟ (صفحة 4 و 11 بالمذكرة / PDF ص 5 و 12)
انظر إلى المشتقتين $p$ و $q$: إذا كانتا مرفوعتين **للأس 1 فقط** ولا يوجد ضرب $p \cdot q$، فالمعادلة على الصورة القياسية للاجرانج:
$$P(x, y, z)\,p + Q(x, y, z)\,q = R(x, y, z)$$
وتُحل دائماً بكتابة **معادلات لاجرانج المساعدة (Lagrange's Auxiliary Equations)**:
$$\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$$
ثم نوجد **ثابتين مستقلين** $u(x, y, z) = a$ و $v(x, y, z) = b$، ويكون الحل العام هو:
$$\Phi(u, v) = 0 \qquad (\text{حيث } \Phi \text{ دالة اختيارية})$$

---

### 🧠 الاستراتيجيات الثلاث في المذكرة لإيجاد الثابتين $a$ و $b$ (PDF ص 13–16)
في الامتحان، عند النظر إلى الكسور $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$، ستواجه إحدى **الحالات الثلاث التالية بالضبط** (وقد وردت جميعها في أمثلة المذكرة من Ex.1 إلى Ex.4):

#### 🔹 الاستراتيجية 1: الفصل المباشر بين كسرين (Ex.1 & Ex.2 — ص 12–13 بالمذكرة / PDF ص 13–14)
عندما يحتوي كسران على متغيرين فقط، نضرب طرفين في وسطين ونكامل مباشرة!
* **مثال 1 (PDF ص 13):** حل $2p - 3q = 2x$:
  * المعاملات: $P = 2,\; Q = -3,\; R = 2x \implies \frac{dx}{2} = \frac{dy}{-3} = \frac{dz}{2x}$.
  * **الكسر الأول والثاني:** $\frac{dx}{2} = \frac{dy}{-3} \implies -3\,dx - 2\,dy = 0 \xrightarrow{\int} -3x - 2y = a$.
  * **الكسر الأول والثالث:** $\frac{dx}{2} = \frac{dz}{2x} \implies x\,dx - dz = 0 \xrightarrow{\int} \frac{x^2}{2} - z = b$.
  * **الحل العام:** $\Phi\left(-3x - 2y,\; \frac{x^2}{2} - z\right) = 0$.
* **مثال 2 (PDF ص 13–14):** حل $\left(\frac{y^2 z}{x}\right)p + xzq = y^2$:
  * المعادلات المساعدة: $\frac{dx}{y^2 z / x} = \frac{dy}{xz} = \frac{dz}{y^2}$.
  * **الكسر الأول والثاني:** نختصر $z$ من المقامين ونضرب طرفين في وسطين: $x^2\,dx - y^2\,dy = 0 \implies x^3 - y^3 = a_1$.
  * **الكسر الأول والثالث:** نختصر $y^2$ من المقامين: $x\,dx - z\,dz = 0 \implies x^2 - z^2 = b_1$.
  * **الحل العام:** $\Phi(x^3 - y^3,\; x^2 - z^2) = 0$.

---

#### 🔹 الاستراتيجية 2: استخدام الثابت الأول $a$ بالتعويض لحل الكسر التالي ثم إرجاعه! (Ex.3 — ص 13–14 بالمذكرة / PDF ص 14–15)
ماذا لو كان الكسر الأخير يحتوي على متغيرات إضافية تمنع فصل المتغيرات مباشرة؟ **نستخدم الثوابت التي أوجدناها للتو لتبديل المتغيرات، وبعد التكامل نُرجع الثوابت إلى قيمها الأصلية!**
* **مثال 3 (PDF ص 14–15):** حل $x\frac{\partial z}{\partial x} + y\frac{\partial z}{\partial y} + t\frac{\partial z}{\partial t} = xyt$:
  * المعادلات المساعدة: $\frac{dx}{x} = \frac{dy}{y} = \frac{dt}{t} = \frac{dz}{xyt}$.
  * **الكسر 1 و 2:** $\frac{dx}{x} = \frac{dy}{y} \implies \ln x - \ln y = \ln a \implies \frac{x}{y} = a$.
  * **الكسر 2 و 3:** $\frac{dy}{y} = \frac{dt}{t} \implies \ln y - \ln t = \ln b \implies \frac{y}{t} = b$.
  * **الكسر 2 و 4:** $\frac{dy}{y} = \frac{dz}{xyt} \implies xt\,dy - dz = 0$. *(لا يمكن تكاملها مباشرة لوجود $x$ و $t$ مع $dy$!)*
  * **الحيلة:** من الثابتين السابقين لدينا $x = a y$ و $t = \frac{y}{b} \implies xt = \frac{a}{b}y^2$. نعوض بها في المعادلة:
    $$\frac{a}{b}y^2\,dy - dz = 0 \xrightarrow{\text{نكامل الآن بسهولة!}} \frac{a}{3b}y^3 - z = c$$
  * **خطوة لا تنسها أبداً:** قبل كتابة الحل العام، أرجع $a = \frac{x}{y}$ و $b = \frac{y}{t}$:
    $$\frac{(x/y)}{3(y/t)}y^3 - z = c \implies \frac{1}{3}xyt - z = c \implies \Phi\left(\frac{x}{y},\; \frac{y}{t},\; \frac{1}{3}xyt - z\right) = 0$$

---

#### 🔹 الاستراتيجية 3: قاعدة تساوي الكسور / المضروبات (القاعدة المؤطرة في ص 14 و Ex.4 ص 15 / PDF ص 15–16)
> 📌 **القاعدة الذهبية في المذكرة (PDF ص 15):**
> *"لأي كسور متساوية، **إذا كان مجموع المقامات يساوي صفراً، فإن مجموع البسوط يساوي صفراً أيضاً!**"*
> وبصورة أعم، إذا ضربنا البسوط والمقامات في معاملات (مضروبات) $(l, m, n)$ بحيث يصبح مجموع المقامات الجديد $lP + mQ + nR = 0$، فإن البسط الجديد يساوي صفراً أيضاً:
> $$l\,dx + m\,dy + n\,dz = 0$$

* **مثال 4 (PDF ص 16 — سؤال امتحانات كلاسيكي هام جداً!):** حل المعادلة:
  $$(y - z)p + (z - x)q = x - y$$
  * **الخطوة 1:** المعادلات المساعدة هي:
    $$\frac{dx}{y - z} = \frac{dy}{z - x} = \frac{dz}{x - y}$$
    *(لاحظ أن كل مقام يحتوي على متغيرين مختلطين، فلا يمكن فصل أي كسرين مباشرة!)*
  * **الخطوة 2 (إيجاد الثابت الأول بجمع المقامات مباشرة):**
    اجمع المقامات الثلاثة: $(y - z) + (z - x) + (x - y) = 0$!
    بما أن مجموع المقامات $= 0$، إذن مجموع البسوط $= 0$:
    $$dx + dy + dz = 0 \xrightarrow{\text{بتكامل الطرفين}} x + y + z = a$$
  * **الخطوة 3 (إيجاد الثابت الثاني بضرب الكسور في $x, y, z$ على الترتيب):**
    اضرب الكسر الأول في $x$ والثاني في $y$ والثالث في $z$ واجمع المقامات:
    $$x(y - z) + y(z - x) + z(x - y) = xy - xz + yz - xy + xz - yz = 0$$
    مجموع المقامات الجديد $= 0$، إذن مجموع البسوط الجديد $= 0$:
    $$x\,dx + y\,dy + z\,dz = 0 \xrightarrow{\text{بتكامل الطرفين}} \frac{x^2}{2} + \frac{y^2}{2} + \frac{z^2}{2} = b$$
  * **الخطوة 4 (الحل العام):**
    $$\Phi\left(x + y + z,\; \frac{x^2}{2} + \frac{y^2}{2} + \frac{z^2}{2}\right) = 0$$

</div>


<div dir="rtl" style="text-align:right">

---
## 3. ✍️ تحدي الورقة والقلم 1 (التمرين 5 — صفحة 16 بالمذكرة / PDF ص 17)

**المسألة (معادلة أويلر):** أوجد الثابتين المستقلين $u(x,y)$ و $v(y,z)$ للمعادلة:
$$x p + y q = z$$

<details>
<summary>💡 <b>تلميح 1:</b> اكتب المعادلات المساعدة</summary>

هنا $P = x, \; Q = y, \; R = z$، إذن:
$$\frac{dx}{x} = \frac{dy}{y} = \frac{dz}{z}$$
</details>

<details>
<summary>💡 <b>تلميح 2:</b> كامل الكسر الأول مع الثاني، ثم الثاني مع الثالث</summary>

$$\int \frac{dx}{x} = \int \frac{dy}{y} \implies \ln|x| - \ln|y| = \ln|a| \implies \ln\left|\frac{x}{y}\right| = \ln|a|$$
وبالمثل للكسر الثاني والثالث مع $y$ و $z$.
</details>

أدخل صيغتي $u$ و $v$ في الخلية التالية:

</div>


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_u = "x/y"
student_v = "y/z"
### END CODE HERE ###

test_lab2.check_paper_challenge_1(student_u, student_v)


<div dir="rtl" style="text-align:right">

---
## 4. 🚀 تحدي الربط بالمشروع: استنتاج معادلة خط انسياب المركبة

لنفترض أن حقل الجهد بالقرب من الهدف يعطى بالدالة:
$$\phi(x, y) = x^2 + 2y^2$$
مركبة المريخ تتحرك باتجاه أشد انحدار للأسفل بسرعة:
$$u = -\frac{\partial\phi}{\partial x} = -2x, \qquad v = -\frac{\partial\phi}{\partial y} = -4y$$
إذن **دالة الانسياب $\psi(x, y)$** التي تبقى ثابتة على مسار المركبة تحقق معادلة لاجرانج:
$$-2x \frac{\partial\psi}{\partial x} - 4y \frac{\partial\psi}{\partial y} = 0$$

**المطلوب على ورقتك:**
استخدم معادلة لاجرانج المساعدة $\frac{dx}{-2x} = \frac{dy}{-4y}$ لإيجاد معادلة منحنى مسار المركبة $y$ بدلالة $x$ والثابت $k$.

<details>
<summary>💡 <b>تلميح 1:</b> بسّط المقامات وفصل المتغيرات</summary>

بقسمة المقامين على $-2$ نحصل على:
$$\frac{dx}{x} = \frac{dy}{2y} \implies 2\frac{dx}{x} = \frac{dy}{y}$$
</details>

<details>
<summary>💡 <b>تلميح 2:</b> كامل الطرفين وخذ الدالة الأسية</summary>

$$2\ln|x| + \ln|k| = \ln|y| \implies \ln(k x^2) = \ln|y| \implies y = k x^2$$
</details>

</div>


In [ ]:
# خلية تقييم: معادلة مسار المركبة
### START CODE HERE ###
rover_streamline = "y = k*x^2"
### END CODE HERE ###

test_lab2.check_paper_challenge_2(rover_streamline)


<div dir="rtl" style="text-align:right">

---
## 5. 💻 من الورقة إلى الكود: متتبع خطوط الانسياب العددي

على تضاريس المريخ الحقيقية لا نملك صيغة جبرية بسيطة لـ $\phi(x,y)$، بل لدينا شبكة أرقام. لذلك يتتبع الحاسوب معادلة لاجرانج المساعدة:
$$\frac{dx}{dt} = u(x,y), \qquad \frac{dy}{dt} = v(x,y)$$
بأخذ خطوات زمنية صغيرة $\Delta t$ (طريقة أويلر الأمامية أو طريقة رونج-كوتا):
$$(x_{\text{new}}, y_{\text{new}}) = (x, y) + \Delta t \cdot (u, v)$$
وهي الفكرة ذاتها التي تنفذها دالة `descend()` على الخلايا الثماني المجاورة في `planners.py`!

أكمل الدالة أدناه لتتبع مسار المركبة عددياً ومقارنته بحلك الورقي $y = 2x^2$ (عند الانطلاق من النقطة $(1, 2)$ حيث $k = y_0/x_0^2 = 2$):

</div>


In [ ]:
# دالة تقييم برمجية: trace_streamline_rk4
def trace_streamline_rk4(velocity_fn, start=(1.0, 2.0), dt=0.01, steps=50):
    """تتبع خط الانسياب (المنحنى المميز للاجرانج) عددياً بخطوة dt."""
    trajectory = [start]
    curr = np.array(start, dtype=float)
    
    ### START CODE HERE ###
    for _ in range(steps):
        vx, vy = velocity_fn(curr[0], curr[1])
        v = np.array([vx, vy], dtype=float)
        curr = curr + dt * v
        trajectory.append(tuple(curr))
    ### END CODE HERE ###
    
    return np.array(trajectory)

test_lab2.test_rk4_streamline_tracer(trace_streamline_rk4)


<div dir="rtl" style="text-align:right">

---
## 6. 🛰️ الرؤية البصرية: تطابق حل لاجرانج الورقي مع المتتبع العددي

شغّل الخلية التالية لترى كيف ينطبق المسار العددي (الخط الأحمر) تماماً فوق المنحنى التحليلي $y = 2x^2$ (الخط المتقطع الأزرق) الذي استنتجته بطريقة لاجرانج على الورق، وكيف يتعامد دائماً مع خطوط تساوي الجهد $\phi = \text{const}$:

</div>


In [ ]:
def sample_velocity(x, y):
    return -2.0 * x, -4.0 * y

traj = trace_streamline_rk4(sample_velocity, start=(1.0, 2.0), dt=0.01, steps=120)

x_vals = np.linspace(-2, 2, 25)
y_vals = np.linspace(-0.5, 2.5, 25)
X, Y = np.meshgrid(x_vals, y_vals)
PHI = X**2 + 2 * Y**2
U, V = sample_velocity(X, Y)

x_exact = np.linspace(0.05, 1.0, 100)
y_exact = 2.0 * (x_exact ** 2)

plt.figure(figsize=(8.5, 6))
plt.contour(X, Y, PHI, levels=12, cmap='viridis', alpha=0.5)
plt.streamplot(X, Y, U, V, color='lightgray', density=0.9)
plt.plot(x_exact, y_exact, 'b--', linewidth=3.5, label=r"Paper Solution (Lagrange): $y = 2x^2$")
plt.plot(traj[:, 0], traj[:, 1], 'r-', linewidth=2.2, label="Numerical Tracer (Code)")
plt.plot(1.0, 2.0, 'go', markersize=9, label="Start (1, 2)")
plt.plot(0.0, 0.0, 'y*', markersize=13, label="Goal (0, 0)")
plt.title("Lagrange Characteristic Curve vs. Numerical Streamline Tracer", fontsize=12)
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

print("📌 ملخص اللاب 2:")
print("  1) المعادلة شبه الخطية P*p + Q*q = R تُحل بمعادلات لاجرانج المساعدة dx/P = dy/Q = dz/R.")
print("  2) المنحنيات المميزة للاجرانج هي نفسها خطوط انسياب حركة المركبة في حقل الجهد.")
print("  3) دالة descend() في planners.py هي التطبيق الشبكي المباشر لتتبع هذه الخطوط من البداية إلى الهدف.")
